## KZSU Zookeeper API Playlist Consolidator

This notebook consolidates all playlists for a given DJ_ID from the KZSU Zookeeper API into a single CSV file conforming to zookeeper CSV upload requirements. Each track is enriched with metadata such as date, time, show name, and other relevant information.

In [2]:
import os
import re

import pandas as pd
import requests

# API configuration
KZSU_API_KEY = os.getenv('KZSU_LIBRARY_API_KEY', 'b3fcd72414bde44530b5fd7b585aaafc08ca41b1')
HEADERS = {
    'X-APIKEY': KZSU_API_KEY,
    'Accept': 'application/vnd.api+json',
    'Content-Type': 'application/vnd.api+json',
}
BASE_URL = 'https://zookeeper.stanford.edu/api/v1/'

# User configuration
DJ_NAME = 'DJ Stace'
DJ_ID = '1428'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def fetch_all_playlists_for_dj(dj_id, page_size=100):
    """Fetch every playlist for a DJ using the site's working search endpoint."""
    playlists = []
    offset = 0

    while True:
        response = requests.get(
            f'{BASE_URL}playlist',
            headers=HEADERS,
            params={
                'filter[airname.id]': str(dj_id),
                'page[size]': page_size,
                'page[offset]': offset,
            },
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        page = payload.get('data', [])
        if isinstance(page, dict):
            page = [page]

        if not page:
            break

        playlists.extend(page)
        if len(page) < page_size:
            break
        offset += len(page)

    return playlists


def consolidate_playlists_for_dj(dj_name, dj_id):
    """Export all spin events from every playlist for the requested DJ."""
    playlists = fetch_all_playlists_for_dj(dj_id)
    if not playlists:
        raise RuntimeError(f'No playlists found for DJ id {dj_id!r}.')

    all_tracks = []
    for playlist in playlists:
        attributes = playlist.get('attributes') or {}
        events = attributes.get('events') or []

        for event in events:
            if event.get('type') != 'spin':
                continue

            all_tracks.append({
                'date': attributes.get('date', ''),
                'time': attributes.get('time', ''),
                'show_name': attributes.get('airname') or attributes.get('name') or '',
                'dj_name': dj_name,
                'artist': event.get('artist', ''),
                'title': event.get('track', '') or event.get('title', ''),
                'album': event.get('album', ''),
                'label': event.get('label', ''),
                'duration': event.get('duration', ''),
            })

    columns = ['date', 'time', 'show_name', 'dj_name', 'artist', 'title', 'album', 'label', 'duration']
    df = pd.DataFrame(all_tracks, columns=columns)
    safe_name = re.sub(r'[^a-z0-9]+', '_', dj_name.lower()).strip('_') or 'dj'
    output_path = os.path.join(OUTPUT_DIR, f'{safe_name}_all_playlists_consolidated.csv')
    df.to_csv(output_path, index=False)
    print(f'Consolidated {len(playlists)} playlists and {len(df)} tracks to {output_path}')
    return df


# Run consolidation
consolidated_df = consolidate_playlists_for_dj(DJ_NAME, DJ_ID)

Consolidated 268 playlists and 7563 tracks to outputs/dj_stace_all_playlists_consolidated.csv
